In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import warnings
warnings.filterwarnings('ignore')



pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)


import os

# List the contents of the /kaggle/input directory to find the exact path of the dataset
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
df = pd.read_csv('/kaggle/input/sales-data-analysis/Sales Data.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.sample(5)

In [ ]:
df.info()

In [ ]:
df.columns

In [ ]:
df.count()

In [ ]:
df.drop(['Unnamed: 0'], axis = 1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
df.shape

In [ ]:
# Check missing values in dataset by heatmap
plt.figure(figsize=(10,6))
sns.heatmap(df.isnull(),cbar=False)
plt.title('Heatmap for Null values')
plt.show()

In [ ]:
df.isnull().sum()

In [ ]:
# check duplicate values in dataset
df.duplicated().sum()

In [ ]:
# See the Duplilcated values
df_cleaned = df[df.duplicated()]
df_cleaned.columns

In [ ]:
# group by all columns except those to be aggregated
columns_to_group_by = ['Product', 'Order Date', 'Purchase Address', 'Month', 'City', 'Hour']


# Aggregae the duplicated rows by summing 'Quantity ordered' and 'sales'
df_aggregated = df.groupby(columns_to_group_by).agg({
    'Quantity Ordered': 'sum',
    'Sales' : 'sum',
    'Price Each' : 'first' # or mean if needed
}).reset_index()

df_aggregated.to_csv('cleaned_dataset.csv', index=False)

print('Duplicates handled and cleaned dataset saved.')

In [ ]:
# Convert Order data dtype into datetime
df['Order Date'] = pd.to_datetime(df['Order Date'])

In [ ]:
df.info()

Exploratory Data Analysis

Univariate Analysis

Univariate analysis involves analyzing the distribution of individual columns.
Descriptive Statistics
: Summary statistics for each column.
Distribution Plots
: Histograms or count plots for each column.

In [ ]:
# df.select_dtypes(include=['int64','float64'])

df_sample = df.sample(n=1000)  # Adjust the sample size as needed
df_sample.select_dtypes(include=['int64', 'float64'])

In [ ]:
# Plotting settings
sns.set(style="whitegrid")

# Distribution plots for numerical columns
numerical_columns = ['Quantity Ordered','Price Each', 'Sales', 'Month', 'Hour']
for column in numerical_columns:
    plt.figure(figsize=(10,6))
    sns.histplot(df[column], kde=True)
    plt.title(f'Distribution of {column}')
    plt.show()

In [ ]:
# Count plots for categorical columns
categorical_columns = ['Product', 'City']
for column in categorical_columns:
    plt.figure(figsize=(10, 6))
    sns.countplot(y=df[column], order=df[column].value_counts().index)
    plt.title(f'Count Plot of {column}')
    plt.show()

Bivariate Analysis

Bivariate analysis involves examining the relationship between two columns.
Correlation Matrix
: Heatmap to see the correlation between numerical features.
Scatter Plots
: For pairs of numerical columns.
Box Plots
: For numerical vs. categorical columns.

In [ ]:
corr_matrix = df.select_dtypes(include=['number']).corr()
corr_matrix

In [ ]:
# correlation matrix
plt.figure(figsize=(12,8))
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm')
plt.title('correlation Matrix')
plt.show()

In [ ]:
# scatter plots for pairs of numerical columns
sns.pairplot(df[numerical_columns])
plt.suptitle('Pairplot of Numerical Columns', y=1.02)
plt.show()

In [ ]:
# Box plots for numerical vs categorical columns
for cat_column in categorical_columns:
    for num_column in numerical_columns:
        plt.figure(figsize=(10,6))
        sns.boxplot(x=cat_column, y=num_column, data=df)
        plt.title(f'Box plot of {num_column} by {cat_column}')
        plt.xticks(rotation=90)
        plt.show()

Multivariate Analysis

Multivariate analysis involves examining relationships among three or more columns simultaneously.
Pair Plot
: With a hue for categorical data.
FacetGrid
: For conditional distributions.
Heatmaps
: For interaction between multiple features.

In [ ]:
df_sample.select_dtypes(include=['object']).head(3)

In [ ]:
# Pair Pplot with hue for a categorical column
sns.pairplot(df, hue = 'Product')
plt.suptitle('Pairplot with product', y=1.02)
plt.show()

Product Category Differentiation:

The coloring by product category helps differentiate how different categories behave with respect to numerical features.
It provides insights into whether certain numerical features are good discriminators for product categories.
By examining these pair plots, we can derive valuable insights into the relationships between features, the influence of product categories, and potential areas for further investigation in our data analysis process.

In [ ]:
# Check the number of unique values
unique_products = df['Product'].nunique()
unique_cities = df['City'].nunique()

In [ ]:
# If there are too many unique values, filter to a manageable subset
if unique_products > 10:
    products_to_plot = df['Product'].value_counts().index[:10]
    df = df[df['Product'].isin(products_to_plot)]

if unique_cities > 5:
    cities_to_plot = df['City'].value_counts().index[:5]
    df = df[df['City'].isin(cities_to_plot)]

# Create the FacetGrid
g = sns.FacetGrid(df, col='City', row='Product', margin_titles=True, height=3)
g.map_dataframe(sns.histplot, 'Sales', kde=True)
g.add_legend()
plt.suptitle('FacetGrid of Sales by Product and City', y=1.02)
plt.show()

In [ ]:
# heatmpa for interaction between multiple features
pivot_table = df.pivot_table(values= 'Sales', index='Product', columns='City', aggfunc = 'sum')
plt.figure(figsize=(12,8))
sns.heatmap(pivot_table, annot=True, fmt='.0f', cmap='YlGnBu')
plt.title('Heatmap of Sales by Product and City')
plt.show()

What We Learn from the Above Heatmap

Sales Distribution Across Cities:
The heatmap shows how sales are distributed across different products and cities.
By examining the color intensity, we can quickly identify cities where certain products perform well (darker shades) or poorly (lighter shades).
High and Low Performing Products and Cities:
Products with consistently high sales across multiple cities can be identified.
Cities where certain products do not sell well are also easily identifiable.
This information can be crucial for understanding market trends and making data-driven business decisions.
Comparative Analysis:
The heatmap allows for easy comparison between different products within the same city.
Similarly, it enables comparison of the same product across different cities.
Identifying Patterns and Trends:
Patterns such as regional preferences for certain products may emerge.
Trends over time can be inferred if the data includes a temporal component (e.g., different months or years).
Overall, this heatmap serves as a powerful visualization tool to understand the interaction between sales, products, and cities, enabling more informed strategic decisions based on data.